# Diagnóstico do ambiente

Execute a célula para selecionar `device` automaticamente e testar PyTorch e TorchVision. Outputs já salvos podem ser de uma execução anterior; execute novamente para obter o estado atual.


In [3]:
import importlib.metadata
import importlib.util
import shutil
import subprocess
import sys

import torch

print("Python:", sys.version.split()[0])
print("Executável:", sys.executable)
print("PyTorch:", torch.__version__)
print("TorchVision:", importlib.metadata.version("torchvision") if importlib.util.find_spec("torchvision") else "não instalado")
print("CUDA da compilação PyTorch:", torch.version.cuda or "não disponível (build CPU)")

if shutil.which("nvidia-smi"):
    resultado = subprocess.run(["nvidia-smi"], text=True, capture_output=True, check=False)
    print("nvidia-smi:\n", resultado.stdout or resultado.stderr)
    if resultado.returncode:
        print("nvidia-smi falhou com código", resultado.returncode)
else:
    print("nvidia-smi não encontrado neste ambiente.")

cuda_disponivel = torch.cuda.is_available()
device = torch.device("cuda" if cuda_disponivel else "cpu")
print("CUDA disponível:", cuda_disponivel)
print("Dispositivo selecionado:", device)
if cuda_disponivel:
    print("GPU:", torch.cuda.get_device_name(0))
    print("Capacidade de computação:", torch.cuda.get_device_capability(0))
    # Uma operação real, seguida de sincronização, verifica a execução no dispositivo.
    amostra = torch.tensor([[1.0, 2.0], [3.0, 4.0]], device=device)
    resultado_cuda = amostra @ amostra
    torch.cuda.synchronize()
    print("Operação CUDA concluída:", resultado_cuda)
else:
    print("Executando em CPU. Em um runtime GPU, confira o kernel, o driver e o build CUDA do PyTorch.")

try:
    import torchvision
    from torchvision.ops import nms
    caixas = torch.tensor([[0.0, 0.0, 1.0, 1.0]], device=device)
    nms(caixas, torch.tensor([0.9], device=device), 0.5)
    if cuda_disponivel:
        torch.cuda.synchronize()
    print("Operador TorchVision no dispositivo: OK")
except Exception as exc:
    print(f"Falha ao validar TorchVision: {type(exc).__name__}: {exc}")
    print("Confira as versões compatíveis de torch e torchvision; nenhuma instalação foi alterada.")


Python: 3.13.18
Executável: /usr/bin/python3
PyTorch: 2.14.1+cpu
TorchVision: 0.29.1+cpu
CUDA da compilação PyTorch: não disponível (build CPU)
nvidia-smi não encontrado neste ambiente.
CUDA disponível: False
Dispositivo selecionado: cpu
Executando em CPU. Em um runtime GPU, confira o kernel, o driver e o build CUDA do PyTorch.
Operador TorchVision no dispositivo: OK


## Dataset

No Colab, monte o Drive e ajuste `DATA_DIR` para a pasta do dataset. Localmente, o caminho padrão é `data/`. A célula não cria nem altera arquivos.


In [ ]:
from pathlib import Path

try:
    from google.colab import drive
except ImportError:
    DATA_DIR = Path("data").resolve()
else:
    drive.mount("/content/drive")
    DATA_DIR = Path("/content/drive/MyDrive/datasets")  # Ajuste para sua pasta no Drive.

print("Dataset:", DATA_DIR)
print("Pasta encontrada:", DATA_DIR.is_dir())
